# Global Happiness & Well Being Analysis


## Data Inspection (Raw Annual Files)

Reviewing shape metrics and column headers for **2017**, **2018**, and **2019** raw CSVs to identify potential schema drifts prior to concatenation.

In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df1=pd.read_csv("../data/raw/2017.csv")
df2=pd.read_csv("../data/raw/2018.csv")
df3=pd.read_csv("../data/raw/2019.csv")

files=[df1,df2,df3]
for file in files:
    rc=file.shape
    print("=" * 20,"Number Of Rows and Columns","="*20)
    print(f"The file contains {rc[0]} rows and {rc[1]} columns")
    print("\n","=" * 20,"Names Of Columns","="*20)
    print(file.columns)

==================== Number Of Rows and Columns ====================
The file contains 155 rows and 12 columns

 ==================== Names Of Columns ====================
Index(['Country', 'Happiness.Rank', 'Happiness.Score', 'Whisker.high',
       'Whisker.low', 'Economy..GDP.per.Capita.', 'Family',
       'Health..Life.Expectancy.', 'Freedom', 'Generosity',
       'Trust..Government.Corruption.', 'Dystopia.Residual'],
      dtype='str')
==================== Number Of Rows and Columns ====================
The file contains 156 rows and 9 columns

 ==================== Names Of Columns ====================
Index(['Overall rank', 'Country or region', 'Score', 'GDP per capita',
       'Social support', 'Healthy life expectancy',
       'Freedom to make life choices', 'Generosity',
       'Perceptions of corruption'],
      dtype='str')
==================== Number Of Rows and Columns ====================
The file contains 156 rows and 9 columns

 ==================== Names Of Columns ===

## Column Renaming & Feature Alignment

Renaming columns across `df1` (2017), `df2` (2018), and `df3` (2019) to ensure seamless row wise concatenation (`pd.concat`) without introducing duplicate or `NaN` columns.

In [2]:
df1=df1.rename(columns={'Happiness.Rank':'Overall rank','Happiness.Score':'Score','Economy..GDP.per.Capita.':'GDP per capita','Health..Life.Expectancy.':'Healthy life expectancy','Trust..Government.Corruption.':'Perceptions of corruption','Family':'Social support'})
df2=df2.rename(columns={'Freedom to make life choices':'Freedom','Country or region':'Country'})
df3=df3.rename(columns={'Freedom to make life choices':'Freedom','Country or region':'Country'})

## Pruning Unnecessary 2017 Features

Dropping confidence interval bounds (`Whisker.*`) and calculation residuals (`Dystopia.Residual`) from `df1` to ensure all annual DataFrames share an identical set of core indicators.

In [3]:
df1=df1.drop(columns=['Whisker.high','Whisker.low','Dystopia.Residual'])

## Injecting Year Columns

Adding `Year` tags (2017, 2018, 2019) to each respective dataset so we can slice, group, and analyze happiness metrics over time once the files are concatenated.

In [4]:
df1['Year']=2017
df2['Year']=2018
df3['Year']=2019

## Final Concatenation & Schema Inspection

Combining all standardized annual datasets into `df` with a clean, reindexed layout. Previewing the top rows to confirm seamless alignment across all features.

In [5]:
df=pd.concat([df1,df2,df3],ignore_index=True)
print(df.shape)
df.head()

(467, 10)


,Country,Overall rank,Score,GDP per capita,Social support,Healthy life expectancy,Freedom,Generosity,Perceptions of corruption,Year
0,Norway,1,7.537,1.616463,1.533524,0.796667,0.635423,0.362012,0.315964,2017
1,Denmark,2,7.522,1.482383,1.551122,0.792566,0.626007,0.355280,0.400770,2017
2,Iceland,3,7.504,1.480633,1.610574,0.833552,0.627163,0.475540,0.153527,2017
3,Switzerland,4,7.494,1.564980,1.516912,0.858131,0.620071,0.290549,0.367007,2017
4,Finland,5,7.469,1.443572,1.540247,0.809158,0.617951,0.245483,0.382612,2017


## Data Quality & Missingness Inspection

Checking for missing values (`NaN`) and schema types across `df`. Filtering the dataset specifically for missing `Perceptions of corruption` records to inform imputation strategy.

In [6]:
print("="*20,"Number Of Null Values","="*20)
print(df.isnull().sum())

print("\n","="*20,"Columns Description","="*20)
print(df.info())

df[df['Perceptions of corruption'].isnull()]

==================== Number Of Null Values ====================
Country                      0
Overall rank                 0
Score                        0
GDP per capita               0
Social support               0
Healthy life expectancy      0
Freedom                      0
Generosity                   0
Perceptions of corruption    1
Year                         0
dtype: int64

 ==================== Columns Description ====================
<class 'pandas.DataFrame'>
RangeIndex: 467 entries, 0 to 466
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Country                    467 non-null    str    
 1   Overall rank               467 non-null    int64  
 2   Score                      467 non-null    float64
 3   GDP per capita             467 non-null    float64
 4   Social support             467 non-null    float64
 5   Healthy life expectancy    467 non-null    float64
 6   Fr

,Country,Overall rank,Score,GDP per capita,Social support,Healthy life expectancy,Freedom,Generosity,Perceptions of corruption,Year
174,United Arab Emirates,20,6.774,2.096,0.776,0.67,0.284,0.186,NaN,2018


## Validating Unique Country-Year Pairs

Calculating the sum of duplicate `(Year, Country)` combinations in `df` to confirm that each country entry is unique per annual survey.

In [7]:
duplicated_countries=df.duplicated(subset=['Year','Country']).sum()
print(f"The duplicate countries in year 2017, 2018, and 2019 are {duplicated_countries}")

The duplicate countries in year 2017, 2018, and 2019 are 0


## Missing Value Imputation For UAE (Year 2018)

Calculating the mean `Perceptions of corruption` value for the United Arab Emirates across 2017 and 2019, then filling the missing 2018 value to restore dataset completeness.

In [34]:
year_17_19=df.loc[(df['Country']=='United Arab Emirates') & ((df['Year']==2017) | (df['Year']==2019)),'Perceptions of corruption'].mean()
df.loc[(df['Country']=='United Arab Emirates') & (df['Year']==2018),'Perceptions of corruption']=year_17_19


## Validating UAE Data Completeness

Filtering for `Country == 'United Arab Emirates'` across all survey years to confirm that the 2018 `Perceptions of corruption` score has been cleanly populated.

In [33]:
df[df['Country']=='United Arab Emirates']

,Country,Overall rank,Score,GDP per capita,Social support,Healthy life expectancy,Freedom,Generosity,Perceptions of corruption,Year
20,United Arab Emirates,21,6.648,1.626343,1.26641,0.726798,0.608345,0.360942,0.324490,2017
174,United Arab Emirates,20,6.774,2.096000,0.77600,0.670000,0.284000,0.186000,0.253245,2018
331,United Arab Emirates,21,6.825,1.503000,1.31000,0.825000,0.598000,0.262000,0.182000,2019


In [42]:
grouped_score=df.groupby('Country')['Score'].mean()
top_10=grouped_score.sort_values(ascending=False).head(10)
top_10

Country
Finland        7.623333
Norway         7.561667
Denmark        7.559000
Iceland        7.497667
Switzerland    7.487000
Netherlands    7.435333
New Zealand    7.315000
Sweden         7.313667
Canada         7.307333
Australia      7.261333
Name: Score, dtype: float64

In [43]:
bottom_10=grouped_score.sort_values(ascending=True).head(10)
bottom_10

Country
Central African Republic    2.953000
Burundi                     3.195000
South Sudan                 3.232667
Tanzania                    3.294333
Rwanda                      3.404333
Yemen                       3.442667
Syria                       3.462000
Afghanistan                 3.543000
Haiti                       3.594000
Botswana                    3.614667
Name: Score, dtype: float64